In [1]:
import Pkg;
Pkg.add("StatsBase")

using LinearAlgebra
using StatsBase

  Installing known registries into `C:\Users\alex1\.julia`
       Added `General` registry to C:\Users\alex1\.julia\registries
    Updating registry at `C:\Users\alex1\.julia\registries\General.toml`
   Resolving package versions...
   Installed LogExpFunctions ───── v0.3.29
   Installed AliasTables ───────── v1.1.3
   Installed DataAPI ───────────── v1.16.0
   Installed IrrationalConstants ─ v0.2.6
   Installed StatsAPI ──────────── v1.8.0
   Installed OrderedCollections ── v1.8.1
   Installed SortingAlgorithms ─── v1.2.2
   Installed StatsBase ─────────── v0.34.9
   Installed Missings ──────────── v1.2.0
   Installed PtrArrays ─────────── v1.3.0
   Installed Statistics ────────── v1.11.1
   Installed DocStringExtensions ─ v0.9.5
   Installed DataStructures ────── v0.19.3
    Updating `C:\Users\alex1\.julia\environments\v1.11\Project.toml`
  [2913bbd2] + StatsBase v0.34.9
    Updating `C:\Users\alex1\.julia\environments\v1.11\Manifest.toml`
  [66dad0bd] + AliasTables v1.1.3
  [9a962f9

In [3]:
using Random, StatsBase

Random.seed!(1234)  # set the global seed

TaskLocalRNG()

# General
This code implements the Approximate Cholesky algorithm in the most naive way. It uses dense matrix operations. We have optimized this for readability. For a full solve, this approximate factorization is then used with PCG to solve a system.

A fast sparse solver (that also implements the full pipeline) is publicly available here:
[Github link: SDD solver](https://rjkyng.github.io/SDDM2023/)

Attached in the email you can see comparisons between the optimized solver and other incomplete cholesky implementations or Hypres BoomerAMG on problems up to 10^9 nonzeros.

## Disclaimer
We have not properly tested this notebook and there might be errors in the implementation. There can also be stability issues depending on the randomness in this implementation. This is not the case in the serious implementation. For further benchmarking please use the public solver above.


In [4]:
# This is general Cholesky

function myChol(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  for i = 1:(n-1)
    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li
    S = S - li*li'
  end
  return fac
end

myChol (generic function with 1 method)

In [5]:
# Consider a clique on n vertices
n = 10
L_Kn = I - ones(n,n)/n

chol_fac = myChol(L_Kn) # Generate Factor

@show opnorm(chol_fac - tril(chol_fac)) # Check that the factorization is lower triangular

@show opnorm(L_Kn-chol_fac*chol_fac') # Check that the factorization equals the input matrix

chol_fac

opnorm(chol_fac - tril(chol_fac)) = 3.0195616480790007e-16
opnorm(L_Kn - chol_fac * chol_fac') = 3.893763993344898e-16


10×9 Matrix{Float64}:
  0.948683   1.47196e-17   1.66905e-17  …   5.09902e-17   8.83177e-17
 -0.105409   0.942809      1.4836e-17       4.53247e-17   7.85046e-17
 -0.105409  -0.117851      0.935414        -7.93182e-17  -1.37383e-16
 -0.105409  -0.117851     -0.133631        -6.7987e-17   -1.17757e-16
 -0.105409  -0.117851     -0.133631         0.0           0.0
 -0.105409  -0.117851     -0.133631     …   4.53247e-17   7.85046e-17
 -0.105409  -0.117851     -0.133631         3.39935e-17   5.88785e-17
 -0.105409  -0.117851     -0.133631         0.816497      0.0
 -0.105409  -0.117851     -0.133631        -0.408248      0.707107
 -0.105409  -0.117851     -0.133631        -0.408248     -0.707107

In [21]:
# This is still normal cholesky, we just rewrite how we update the schur complement
function myChol2(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  for i = 1:(n-1)
    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li
    # Same up to here

    # Remove the star...
    ai = -S[:,i]
    di = S[i,i]
    ai[i] = 0

    S_noi = copy(S)
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    S_noi -= Diagonal(ai)

    S = S_noi + Diagonal(ai)-ai*ai'/di # ...and insert the clique

  end
  return fac
end

myChol2 (generic function with 1 method)

In [22]:
# Same example with the second, equivalent Cholesky code
n = 10
L_Kn = I - ones(n,n)/n

chol_fac = myChol2(L_Kn) # Generate Factor

@show opnorm(chol_fac - tril(chol_fac)) # Check that the factorization is lower triangular

@show opnorm(L_Kn-chol_fac*chol_fac') # Check that the factorization equals the input matrix

chol_fac

opnorm(chol_fac - tril(chol_fac)) = 0.0
opnorm(L_Kn - chol_fac * chol_fac') = 6.794445557167506e-16


10×9 Matrix{Float64}:
  0.948683   0.0        0.0       …   0.0        0.0        0.0
 -0.105409   0.942809   0.0           0.0        0.0        0.0
 -0.105409  -0.117851   0.935414      0.0        0.0        0.0
 -0.105409  -0.117851  -0.133631      0.0        0.0        0.0
 -0.105409  -0.117851  -0.133631      0.0        0.0        0.0
 -0.105409  -0.117851  -0.133631  …   0.0        0.0        0.0
 -0.105409  -0.117851  -0.133631      0.866025   0.0        0.0
 -0.105409  -0.117851  -0.133631     -0.288675   0.816497   0.0
 -0.105409  -0.117851  -0.133631     -0.288675  -0.408248   0.707107
 -0.105409  -0.117851  -0.133631     -0.288675  -0.408248  -0.707107

In [10]:
# Now the simple aprox-chol:

# Sampling rule for a single edge
function cliqueSample(a)
  nbrs = findall(a .> 1e-12)
  cliqueSampleLap = zeros(n,n)
  for j in nbrs
    k = sample(1:n,Weights(a))
    if k != j
      wjk = a[j]*a[k]/(a[j]+a[k])
      bjk = zeros(n); bjk[j] = 1; bjk[k] = -1;
      cliqueSampleLap += wjk*bjk*bjk'
    end
  end
  return cliqueSampleLap
end

cliqueSample (generic function with 1 method)

In [16]:
function cliqueSample_sdd(a,i)
  self_loop= sum(abs,a)-a[i]
  a[i]=0
  nbrs = findall(a .> 1e-12)
  cliqueSampleLap = zeros(n,n)
  cliqueSampleLap[diagind(cliqueSampleLap)] .+= self_loop* a
  for j in nbrs
    k = sample(1:n,Weights(a))
    if k != j
      wjk = abs(a[j]*a[k])/(abs(a[j])+ abs(a[k]))
      bjk = zeros(n); bjk[j] = 1; bjk[k] = -sign(a[k]* a[j]);
      cliqueSampleLap += wjk*bjk*bjk'
    end
  end
  return cliqueSampleLap
end

cliqueSample_sdd (generic function with 1 method)

In [11]:
# This is the excat same algorithm as just with sampling
function apxChol1(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  connected = true
  for i = 1:(n-1)
    if S[i,i] < 1e-12 # If the diagonal is zero, then the row is already zeroed out
      if connected
        println("The factorization disconnected the graph, anything can happen now!")
        connected = false
      end

      continue
    end

    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    # Remove the star...
    ai = -S[:,i]
    di = S[i,i]
    ai[i] = 0

    S_noi = copy(S)
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    S_noi -= Diagonal(ai)

    S = S_noi + cliqueSample(ai) # Only difference: "Diagonal(ai)-ai*ai'/di" -> "cliqueSample(ai)"
  end
  return fac
end

apxChol1 (generic function with 1 method)

In [ ]:
function erase_self_loops(A::AbstractMatrix, B::AbstractMatrix)
    m, n = size(A)
    @assert size(B) == (m, n) "Matrices A and B must have the same dimensions"

    # Initialize C with zeros (efficient allocation)
    # Use the same type as the input matrices (e.g., Float64)
    C = zeros(promote_type(eltype(A), eltype(B)), m, n)

    @inbounds for j in 1:n
        for i in 1:m
            valA = A[i, j]
            valB = B[i, j]

            # Check if signs are different
            # Note: sign(0) is 0, so 0 and 5 are considered "different signs" here.
            # If you consider 0 to be "same sign" as positives, change this logic.
            println("This shouldn't be called if it is a Laplacian")

            
            # 1. Base assignment
            C[i, j] = valA + valB
            
            # 2. Calculate the penalty term
            # Term = 2 * (|A| + |B| - |A+B|)
            term = 2 * (abs(valA) + abs(valB) - abs(valA + valB))
            
            # 3. Add to diagonals
            C[i, i] += term
            C[j, j] += term 
            # If signs are the same, "nothing happens" (C[i,j] remains 0)
        end
    end

    return C
end

erase_self_loops (generic function with 1 method)

In [23]:
function apxChol_sdd1(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  connected = true
  for i = 1:(n-1)
    if S[i,i] < 1e-12 # If the diagonal is zero, then the row is already zeroed out
      if connected
        println("The factorization disconnected the graph, anything can happen now!")
        connected = false
      end

      continue
    end

    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    # Remove the star...
    ai = -S[:,i]
    di = S[i,i]
    #ai[i] = 0

    S_noi = copy(S)
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    S_noi -= Diagonal(ai)

    S = erase_self_loops(S_noi,cliqueSample_sdd(ai,i))
  end
  return fac
end

apxChol_sdd1 (generic function with 1 method)

In [18]:
# Same example
n = 10
L_Kn = I - ones(n,n)/n

fApxChol = apxChol1(L_Kn)
rel_eigs = eigvals((L_Kn + ones(n,n) / n)*inv(fApxChol*fApxChol' + ones(n,n) / n))

# The ratio of the smallest to the largest eigenvalue
maximum(rel_eigs) / minimum(rel_eigs)

2.6696171544856893

In [24]:
# Same example
n = 10
L_Kn = I - ones(n,n)/n

fApxChol = apxChol_sdd1(L_Kn)
rel_eigs = eigvals((L_Kn + ones(n,n) / n)*inv(fApxChol*fApxChol' + ones(n,n) / n))

# The ratio of the smallest to the largest eigenvalue
maximum(rel_eigs) / minimum(rel_eigs)

This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is a Laplacian
This shouldn't be called if it is 

SingularException: SingularException(3)

In [26]:
# Now we allow multiedges for the sampling
function cliqueSampleMultiEdge(a, multiplicity)
  nbrs = findall(a .> 1e-12)
  n = size(a)[1]
  cliqueSampleLap = zeros(n,n)
  new_multiedges = zeros(Int, n,n)
  for j in nbrs
    for rep in 1:multiplicity[j]
      k = sample(1:n,Weights(a))
      if k != j
        multiWeightJ = a[j] / multiplicity[j]
        multiWeightk = a[k]
        wjk =  multiWeightJ * a[k] / (a[j] + a[k])
        new_multiedges[j,k] += 1; new_multiedges[k,j] += 1;
        bjk = zeros(n); bjk[j] = 1; bjk[k] = -1;
        cliqueSampleLap += wjk*bjk*bjk'
      end
    end
  end
  return cliqueSampleLap, new_multiedges
end

cliqueSampleMultiEdge (generic function with 1 method)

In [27]:
# We need a datastructure that holds the number of multiedges, this is the most naive way to do it
# Note that multiedges == 1 is not equal to no multiedges.
function initMultiedges(L, nr_multi)
  multiedges = zeros(Int, n,n,)
  A_mask = -L + Diagonal(diag(L)) .> 0
  multiedges[A_mask] .= nr_multi
  return multiedges
end

initMultiedges (generic function with 1 method)

In [28]:
# This is still the same algorithm and some index handling
function apxCholMulti(L, nr_multi)
  multi_edges = initMultiedges(L, nr_multi)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  for i = 1:(n-1)
    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    ai = -S[:,i]
    di = S[i,i]
    ai[i] = 0

    # S without star on i
    S_noi = copy(S)
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    S_noi -= Diagonal(ai)
    sampled_clique, new_multiedges = cliqueSampleMultiEdge(ai, multi_edges[:,i])
    S = S_noi + sampled_clique
    multi_edges += new_multiedges
  end
  return fac
end

apxCholMulti (generic function with 1 method)

In [29]:
# Same example
n = 10
L_Kn = I - ones(n,n)/n

fApxChol = apxCholMulti(L_Kn, 1) # Second param specifies the number of multiedges
rel_eigs = eigvals((L_Kn + ones(n,n) / n)*inv(fApxChol*fApxChol' + ones(n,n) / n))

# The ratio of the smallest to the largest eigenvalue
maximum(rel_eigs) / minimum(rel_eigs)

1.5266606348318337

In [30]:
# Now the best sampling rule, where we always get a tree
function cliqueSampleTree(a)
  d = sum(a)
  a_c = copy(a)
  nbrs = findall(a .> 1e-12)
  cliqueSampleLap = zeros(n,n)
  for j in nbrs[1:end-1]
    a_c[j] = 0
    k = sample(1:n,Weights(a_c))
    if k != j
      wjk = sum(a_c)*a[j] / d
      bjk = zeros(n); bjk[j] = 1; bjk[k] = -1;
      cliqueSampleLap += wjk*bjk*bjk'
    end
  end
return cliqueSampleLap
end

cliqueSampleTree (generic function with 1 method)

In [31]:
function apxCholTree(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  for i = 1:(n-1)
    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    # Remove the star...
    ai = -S[:,i]
    di = S[i,i]
    ai[i] = 0

    S_noi = copy(S)
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    S_noi -= Diagonal(ai)

    S = S_noi + cliqueSampleTree(ai)
  end
  return fac
end

apxCholTree (generic function with 1 method)

In [32]:
# Same example
n = 20
L_Kn = I - ones(n,n)/n

fApxChol = apxCholTree(L_Kn)
rel_eigs = eigvals((L_Kn + ones(n,n) / n)*inv(fApxChol*fApxChol' + ones(n,n) / n))

# The ratio of the smallest to the largest eigenvalue
println(maximum(rel_eigs) / minimum(rel_eigs))


2.590917716560838


On these small graphs all versions (cliqueSimple are decent, lets make the graphs more challenging, this is a barbell graph with a long path


In [33]:
B = 10
n = 3*B
A_barbell = zeros(n,n)
A_barbell[1:B,1:B] = ones(B,B) / B
A_barbell[2B+1:3B,2B+1:3B] = ones(B,B) / B

A_barbell[B+1:2B, B+1:2B] += diagm(-1 => ones(B-1))
A_barbell[B+1:2B, B+1:2B] += diagm(1 => ones(B-1))
A_barbell[B+1,B] = 1
A_barbell[B,B+1] = 1
A_barbell[2B+1,2B] = 1
A_barbell[2B,2B+1] = 1
L_barbell = -A_barbell + Diagonal(vec(sum(A_barbell,dims=1)))
L_barbell



30×30 Matrix{Float64}:
  0.9  -0.1  -0.1  -0.1  -0.1  -0.1  …   0.0   0.0   0.0   0.0   0.0   0.0
 -0.1   0.9  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1   0.9  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1   0.9  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1   0.9  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1   0.9  …   0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0  …   0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  

In [34]:
# Here the iid sampling
fApxChol_iid = apxChol1(L_barbell)
rel_eigs_iid = eigvals(Symmetric((L_barbell + ones(n,n) / n)*inv(fApxChol_iid*fApxChol_iid' + ones(n,n) / n)))


# The ratio of the smallest to the largest eigenvalue
maximum(rel_eigs_iid) / minimum(rel_eigs_iid)


4.678794202415577

In [36]:
# Here the Tree version
fApxChol_tree = apxCholTree(L_barbell)
rel_eigs_tree = eigvals(Symmetric((L_barbell + ones(n,n) / n)*inv(fApxChol_tree*fApxChol_tree' + ones(n,n) / n)))

maximum(rel_eigs_tree) / minimum(rel_eigs_tree)

3.3907579497416154

In [37]:
# We still don't really break the easy version. Here a star graph, which should be very hard for the easy version
n = 10
L_star = zeros(n,n)+ I
L_star[1,:] .= -1
L_star[:,1] .= -1
L_star[1,1] = n-1
L_star

10×10 Matrix{Float64}:
  9.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
 -1.0   1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
 -1.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
 -1.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0   0.0
 -1.0   0.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0
 -1.0   0.0   0.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0
 -1.0   0.0   0.0   0.0   0.0   0.0   1.0   0.0   0.0   0.0
 -1.0   0.0   0.0   0.0   0.0   0.0   0.0   1.0   0.0   0.0
 -1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   1.0   0.0
 -1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   1.0

In [38]:
# Here the iid sampling
repeats = 10
for i in 1:repeats
  fApxChol_iid = apxChol1(L_star)
  rel_eigs_iid = eigvals(Symmetric((L_star + ones(n,n) / n)*pinv(fApxChol_iid*fApxChol_iid' + ones(n,n) / n)))


  # The ratio of the smallest to the largest eigenvalue
  println("Run id: ", i, " ", maximum(rel_eigs_iid) / minimum(rel_eigs_iid))
end

The factorization disconnected the graph, anything can happen now!
Run id: 1 -2.3197008571156596e16
The factorization disconnected the graph, anything can happen now!
Run id: 2 -2.4478948403248504e16
The factorization disconnected the graph, anything can happen now!
Run id: 3 2.941218981967672e16
The factorization disconnected the graph, anything can happen now!
Run id: 4 2.1341744328572932e16
The factorization disconnected the graph, anything can happen now!
Run id: 5 -7.17190597465392e15
The factorization disconnected the graph, anything can happen now!
Run id: 6 -3.683675178147243e15
The factorization disconnected the graph, anything can happen now!
Run id: 7 -5.70819624471232e15
The factorization disconnected the graph, anything can happen now!
Run id: 8 1.2931199465454802e17
The factorization disconnected the graph, anything can happen now!
Run id: 9 -7.059563915266703e15
The factorization disconnected the graph, anything can happen now!
Run id: 10 -4.709807212674867e16


In [40]:
# Here the Tree version
repeats = 10
for i in 1:repeats
  fApxChol_tree = apxCholTree(L_star)
  rel_eigs_tree = eigvals(Symmetric((L_star + ones(n,n) / n)*inv(fApxChol_tree*fApxChol_tree' + ones(n,n) / n)))
  println("Run id: ", i, " ", maximum(rel_eigs_tree) / minimum(rel_eigs_tree))
end


Run id: 1 38.82404604729034
Run id: 2 51.07461792813175
Run id: 3 39.9501801130999
Run id: 4 58.982599589041214
Run id: 5 27.72342466765727
Run id: 6 53.82726559812903
Run id: 7 49.888885619580904
Run id: 8 48.79519725674101
Run id: 9 43.42715067397168
Run id: 10 44.68995377277255


# Final words
Of course this naive version should not be used in practice. We have now optimized it for understandability. Instead, there is a fast implementation using sparse datastructures and other optimizations. You can access the serious implementation here:

[Github link: ApproxChol SDD solver](https://rjkyng.github.io/SDDM2023/)

This also includes a multi-edge version of cliqueSampleTree, but, the single-sample version is generally enough.

If you have any questions about this code or the serious implementation you can send them to

ybaumann@ethz.ch
and/or
kyng@inf.ethz.ch 



